# Project Nano-Myo
## Notebook 05 - Evaluation


## Step 1 - Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Imports


In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Nano_Myo'
FEATURE_DIR = PROJECT_ROOT / 'features'
MODEL_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SCHEMA_VERSION = 'e2_only_80feat_wl_ssc_rest25000_v1'
INPUT_DIM = 80
NUM_CLASSES = 10

TFLITE_MODEL_PATH = MODEL_DIR / 'nano_myo.tflite'
FINAL_RESULTS_PATH = RESULTS_DIR / 'final_results.txt'
STANDARD_CM_PATH = RESULTS_DIR / 'confusion_matrix.png'
FUNCTIONAL_CM_PATH = RESULTS_DIR / 'functional_confusion_matrix.png'


## Step 3 - Load Artifacts


In [ ]:
paths = {
    'X_test': FEATURE_DIR / f'X_test_{FEATURE_SCHEMA_VERSION}.npy',
    'y_test': FEATURE_DIR / f'y_test_{FEATURE_SCHEMA_VERSION}.npy',
    'subjects_test': FEATURE_DIR / f'subjects_test_{FEATURE_SCHEMA_VERSION}.npy',
    'source_labels_test': FEATURE_DIR / f'source_labels_test_{FEATURE_SCHEMA_VERSION}.npy',
    'metadata': FEATURE_DIR / f'nano_myo_features_holdout_s9_s10_{FEATURE_SCHEMA_VERSION}_metadata.json',
    'tflite': TFLITE_MODEL_PATH,
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

with open(paths['metadata'], 'r', encoding='utf-8') as f:
    metadata = json.load(f)

if metadata['feature_schema_version'] != FEATURE_SCHEMA_VERSION:
    raise ValueError(metadata['feature_schema_version'])
if metadata['exercise_used'] != 'E2':
    raise ValueError(metadata['exercise_used'])
if metadata['input_dim'] != INPUT_DIM:
    raise ValueError(metadata['input_dim'])

X_test = np.load(paths['X_test']).astype(np.float32)
y_test = np.load(paths['y_test']).astype(np.int64)
subjects_test = np.load(paths['subjects_test'], allow_pickle=True)
source_labels_test = np.load(paths['source_labels_test']).astype(np.int64)
TARGET_LABELS = {int(key): value for key, value in metadata['target_labels'].items()}

print(f'X_test: {X_test.shape}')
print(f'Test counts: {Counter(y_test.tolist())}')
print(f'Test subjects: {sorted(np.unique(subjects_test).tolist())}')


## Step 4 - TFLite Inference


In [ ]:
def run_tflite_classifier(tflite_path, X):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_index = input_details['index']
    output_index = output_details['index']
    input_scale, input_zero_point = input_details['quantization']
    output_scale, output_zero_point = output_details['quantization']
    predictions = []

    for row in X.astype(np.float32):
        sample = row.reshape(1, INPUT_DIM)
        if input_details['dtype'] in (np.int8, np.uint8):
            sample = np.round(sample / input_scale + input_zero_point)
            sample = np.clip(sample, np.iinfo(input_details['dtype']).min, np.iinfo(input_details['dtype']).max)
            sample = sample.astype(input_details['dtype'])
        else:
            sample = sample.astype(input_details['dtype'])

        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_index)
        if output_details['dtype'] in (np.int8, np.uint8):
            output = (output.astype(np.float32) - output_zero_point) * output_scale
        predictions.append(int(np.argmax(output, axis=1)[0]))

    return np.asarray(predictions, dtype=np.int64)


y_pred = run_tflite_classifier(TFLITE_MODEL_PATH, X_test)
standard_accuracy = float(np.mean(y_pred == y_test))
tflite_size_bytes = TFLITE_MODEL_PATH.stat().st_size

print(f'Standard accuracy: {standard_accuracy:.4f}')
print(f'TFLite size: {tflite_size_bytes:,} bytes')
print(f'Under 16KB: {tflite_size_bytes < 16 * 1024}')


## Step 5 - Standard Metrics


In [ ]:
labels = list(TARGET_LABELS.keys())
display_labels = [TARGET_LABELS[key] for key in labels]

per_class_accuracy = {}
for class_id, class_name in TARGET_LABELS.items():
    mask = y_test == class_id
    per_class_accuracy[class_id] = float(np.mean(y_pred[mask] == y_test[mask])) if np.any(mask) else float('nan')
    print(f'{class_id}: {class_name}: {per_class_accuracy[class_id]:.4f}')

print(classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=display_labels,
    digits=4,
    zero_division=0,
))


## Step 6 - Confusion Matrix


In [ ]:
standard_cm = confusion_matrix(y_test, y_pred, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(standard_cm, display_labels=display_labels).plot(
    include_values=True,
    cmap='Blues',
    ax=ax,
    xticks_rotation=45,
    values_format='.2f',
)
plt.title('TFLite Confusion Matrix')
plt.tight_layout()
plt.savefig(STANDARD_CM_PATH, dpi=200, bbox_inches='tight')
plt.show()


## Step 7 - Functional Metrics


In [ ]:
cost_matrix = np.array([
    [0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    [0.8, 0.0, 1.0, 0.4, 0.4, 0.6, 0.6, 0.3, 0.3, 0.4],
    [0.8, 1.0, 0.0, 0.3, 0.3, 0.6, 0.6, 0.3, 0.3, 0.3],
    [0.8, 0.4, 0.3, 0.0, 0.2, 0.5, 0.5, 0.3, 0.3, 0.2],
    [0.8, 0.4, 0.3, 0.2, 0.0, 0.5, 0.5, 0.3, 0.3, 0.2],
    [0.8, 0.5, 0.5, 0.4, 0.4, 0.0, 1.0, 0.4, 0.4, 0.4],
    [0.8, 0.5, 0.5, 0.4, 0.4, 1.0, 0.0, 0.4, 0.4, 0.4],
    [0.8, 0.3, 0.3, 0.3, 0.3, 0.4, 0.4, 0.0, 0.6, 0.3],
    [0.8, 0.3, 0.3, 0.3, 0.3, 0.4, 0.4, 0.6, 0.0, 0.3],
    [0.8, 0.4, 0.3, 0.2, 0.2, 0.4, 0.4, 0.3, 0.3, 0.0],
], dtype=np.float32)

sample_costs = cost_matrix[y_test, y_pred]
functional_accuracy = float(1.0 - np.mean(sample_costs))
dangerous_confusion_rate = float(np.mean(sample_costs == 1.0))

print(f'Functional accuracy: {functional_accuracy:.4f}')
print(f'Dangerous confusion rate: {dangerous_confusion_rate:.4f}')


## Step 8 - Functional Confusion Matrix


In [ ]:
functional_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.float32)
for true_label, pred_label in zip(y_test, y_pred):
    functional_cm[true_label, pred_label] += cost_matrix[true_label, pred_label]

row_sums = functional_cm.sum(axis=1, keepdims=True)
functional_cm_normalized = np.divide(
    functional_cm,
    row_sums,
    out=np.zeros_like(functional_cm),
    where=row_sums != 0,
)

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(functional_cm_normalized, display_labels=display_labels).plot(
    include_values=True,
    cmap='Reds',
    ax=ax,
    xticks_rotation=45,
    values_format='.2f',
)
plt.title('Cost-Weighted Confusion Matrix')
plt.tight_layout()
plt.savefig(FUNCTIONAL_CM_PATH, dpi=200, bbox_inches='tight')
plt.show()


## Step 9 - Subject Metrics


In [ ]:
subject_metrics = {}
for subject in sorted(np.unique(subjects_test)):
    mask = subjects_test == subject
    subject_costs = cost_matrix[y_test[mask], y_pred[mask]]
    subject_metrics[str(subject)] = {
        'standard_accuracy': float(np.mean(y_pred[mask] == y_test[mask])),
        'functional_accuracy': float(1.0 - np.mean(subject_costs)),
        'dangerous_confusion_rate': float(np.mean(subject_costs == 1.0)),
        'windows': int(mask.sum()),
    }
    print(subject, subject_metrics[str(subject)])


## Step 10 - Hardware Estimate


In [ ]:
weight_macs = 80 * 64 + 64 * 32 + 32 * 10
estimated_cycles = weight_macs * 4
estimated_inference_ms_at_48mhz = estimated_cycles / 48_000_000 * 1000
estimated_activation_ram_bytes = 80 + 64 + 32 + 10

print(f'MACs per inference: {weight_macs:,}')
print(f'Estimated cycles: {estimated_cycles:,}')
print(f'Estimated inference time at 48MHz: {estimated_inference_ms_at_48mhz:.3f} ms')
print(f'Estimated activation RAM: {estimated_activation_ram_bytes:,} bytes')


## Step 11 - Save


In [ ]:
lines = [
    'Project Nano-Myo final evaluation',
    f'feature_schema_version: {FEATURE_SCHEMA_VERSION}',
    f'tflite_size_bytes: {tflite_size_bytes}',
    f'under_16kb: {tflite_size_bytes < 16 * 1024}',
    f'standard_accuracy: {standard_accuracy:.6f}',
    f'functional_accuracy: {functional_accuracy:.6f}',
    f'dangerous_confusion_rate: {dangerous_confusion_rate:.6f}',
    f'macs_per_inference: {weight_macs}',
    f'estimated_cycles: {estimated_cycles}',
    f'estimated_inference_ms_at_48mhz: {estimated_inference_ms_at_48mhz:.6f}',
    f'estimated_activation_ram_bytes: {estimated_activation_ram_bytes}',
    '',
    'per_class_accuracy:',
]

for class_id, class_name in TARGET_LABELS.items():
    lines.append(f'{class_id}: {class_name}: {per_class_accuracy[class_id]:.6f}')

lines.extend(['', 'subject_metrics:'])
for subject, values in subject_metrics.items():
    lines.append(f'{subject}: {values}')

FINAL_RESULTS_PATH.write_text('\n'.join(lines), encoding='utf-8')

print(f'Final results saved: {FINAL_RESULTS_PATH}')
print(f'Standard confusion matrix saved: {STANDARD_CM_PATH}')
print(f'Functional confusion matrix saved: {FUNCTIONAL_CM_PATH}')
